# IEOR E4004 — Project I: Eliminating Child Care Deserts in NYC
# Complete Solution: Task 1 (Budgeting) + Task 2 (Realistic Expansion & Location)

---

This notebook contains the **full solution** to both halves of the project:

- **PART A — The Budgeting Problem (Task 1):** Idealistic scenario. New facilities can be built anywhere, expansions up to 120% / 500-slot cap, single linear cost tier.
- **PART B — Realistic Capacity Expansion & Location (Task 2):** Expansion capped at 20% with 3-tier costs, new facilities only at approved locations, 0.06-mile minimum distance between all facilities.

Both parts share the same data cleaning pipeline (Sections 1–3), then diverge.


---
# SHARED SECTIONS (used by both Task 1 and Task 2)
---
## 1. Imports


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# pandas: data tables    numpy: math/arrays
# We'll use greedy optimization (no external solver needed).
print("Libraries loaded.")


Libraries loaded.


## 2. Load and Clean All Data

In [ ]:
# Load the 5 raw CSV files
avg_income_nyc           = pd.read_csv('avg_individual_income_nyc.csv', index_col=0)
child_care_regulated_nyc = pd.read_csv('child_care_regulated_nyc.csv', index_col=0)
employment_rate_nyc      = pd.read_csv('employment_rate_nyc.csv', index_col=0)
population_nyc_raw       = pd.read_csv('population_nyc.csv', index_col=0)
potential_locations_nyc  = pd.read_csv('potential_locations_nyc.csv', index_col=0)

# avg_income_nyc: average income per ZIP (179 ZIPs)
# child_care_regulated_nyc: every existing facility (7,740 rows)
# employment_rate_nyc: employment rate per ZIP (179 ZIPs)
# population_nyc_raw: population by age per ZIP (211 ZIPs — needs cleaning)
# potential_locations_nyc: candidate building sites (31,100 rows — Task 2 only)

print(f"Income:              {len(avg_income_nyc)} rows")
print(f"Facilities:          {len(child_care_regulated_nyc)} rows")
print(f"Employment:          {len(employment_rate_nyc)} rows")
print(f"Population (raw):    {len(population_nyc_raw)} rows")
print(f"Potential locations: {len(potential_locations_nyc)} rows")


Income:              179 rows
Facilities:          7740 rows
Employment:          179 rows
Population (raw):    211 rows
Potential locations: 31100 rows


In [ ]:
# --- ZIP remapping dictionary ---
# Many NYC ZIPs are commercial/PO-box codes that have no population data.
# We map each to the residential ZIP that physically contains it.
# (Built in partner's EDA_2, Sections 4.1–4.3)

building_zip_mapping = {
    10020: 10036, 10103: 10036, 10110: 10036, 10111: 10036, 10112: 10036,
    10115: 10025,
    10119: 10001, 10173: 10001, 10199: 10001,
    10154: 10017, 10165: 10017, 10167: 10017,
    10168: 10017, 10169: 10017, 10174: 10017, 10177: 10017,
    10152: 10022, 10153: 10022, 10170: 10022, 10171: 10022, 10172: 10022,
    10271: 10005, 10278: 10007, 10279: 10007,
    10311: 10314,
    11359: 11360, 11371: 11369, 11424: 11432, 11451: 11432,
    11430: 11413, 11439: 11366,
}
# E.g., 10020 (Rockefeller Center) -> 10036 (Midtown West residential ZIP)
print(f"ZIP remappings: {len(building_zip_mapping)}")


ZIP remappings: 31


In [ ]:
# --- Clean population data ---
# Remap commercial ZIPs, then sum populations that merge into the same ZIP.

population_nyc_raw['zipcode'] = population_nyc_raw['zipcode'].astype(int)
population_nyc_raw['zipcode'] = population_nyc_raw['zipcode'].map(
    lambda z: building_zip_mapping.get(z, z)
)
# Replace each commercial ZIP with its residential parent.

numeric_cols = [c for c in population_nyc_raw.columns if c != 'zipcode']
population_nyc = population_nyc_raw.groupby('zipcode')[numeric_cols].sum().reset_index()
# Group by the remapped ZIP and sum. E.g., if 10020 had 200 kids and 10036
# had 3000, after merge 10036 now has 3200.

print(f"Population after cleaning: {len(population_nyc)} ZIPs")


Population after cleaning: 180 ZIPs


In [ ]:
# --- Haversine distance function ---
# Used by both tasks for geographic calculations.

def haversine(lat1, lon1, lat2, lon2):
    """Great-circle distance in miles between two points on Earth."""
    R = 3958.8  # Earth radius in miles
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlam/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# Same formula as partner's EDA_2 Section 4.3.
print("Haversine function defined.")


Haversine function defined.


In [ ]:
# --- Remap potential locations ZIPs ---
# Apply manual mapping, then nearest-ZIP for the rest.

pop_zips = set(population_nyc['zipcode'].astype(int))

potential_locations_nyc['zipcode'] = (
    potential_locations_nyc['zipcode'].astype(int)
    .map(lambda z: building_zip_mapping.get(z, z))
)

still_bad = set(potential_locations_nyc['zipcode'].astype(int)) - pop_zips

if len(still_bad) > 0:
    valid_locs = potential_locations_nyc[
        potential_locations_nyc['zipcode'].astype(int).isin(pop_zips)
    ]
    pop_centroids = valid_locs.groupby('zipcode')[['latitude','longitude']].mean()
    # Average lat/lon of all potential sites in each valid ZIP = ZIP center.

    auto_remap = {}
    for bad_z in still_bad:
        locs = potential_locations_nyc[potential_locations_nyc['zipcode'] == bad_z]
        if len(locs) == 0:
            continue
        clat, clon = locs['latitude'].mean(), locs['longitude'].mean()
        dists = haversine(clat, clon,
                          pop_centroids['latitude'].values,
                          pop_centroids['longitude'].values)
        auto_remap[bad_z] = int(pop_centroids.index[np.argmin(dists)])
    # For each bad ZIP, find the geographically closest valid ZIP.

    potential_locations_nyc['zipcode'] = (
        potential_locations_nyc['zipcode'].astype(int)
        .map(lambda z: auto_remap.get(z, z))
    )

potential_locations_nyc = potential_locations_nyc[
    potential_locations_nyc['zipcode'].astype(int).isin(pop_zips)
].copy()

print(f"Potential locations aligned: {len(potential_locations_nyc)} rows, "
      f"unresolved: {len(set(potential_locations_nyc['zipcode'].astype(int)) - pop_zips)}")


Potential locations aligned: 31100 rows, unresolved: 0


In [ ]:
# --- Fill missing coordinates in child care facilities ---
# 169 facilities lack lat/lon. Fill with their ZIP's centroid.

zip_coords = potential_locations_nyc.groupby('zipcode')[['latitude','longitude']].mean().to_dict('index')

missing_mask = child_care_regulated_nyc['latitude'].isna()
print(f"Facilities missing coords: {missing_mask.sum()}")

for idx in child_care_regulated_nyc[missing_mask].index:
    z = int(child_care_regulated_nyc.loc[idx, 'zipcode'])
    z = building_zip_mapping.get(z, z)
    coords = zip_coords.get(z)
    if coords:
        child_care_regulated_nyc.loc[idx, 'latitude'] = coords['latitude']
        child_care_regulated_nyc.loc[idx, 'longitude'] = coords['longitude']

print(f"Still missing after fill: {child_care_regulated_nyc['latitude'].isna().sum()}")


Facilities missing coords: 169
Still missing after fill: 0


## 3. Compute Child Care Desert Status (shared by both tasks)

In [ ]:
# --- Build master ZIP-level table ---

zipdata = population_nyc[['zipcode']].copy()
zipdata['pop_0_5']  = population_nyc['-5'].values          # children under 5
zipdata['pop_0_12'] = population_nyc['-5'].values + population_nyc['6-12'].values  # children 0-12

zipdata = zipdata.merge(avg_income_nyc[['zipcode','average income']], on='zipcode', how='left')
zipdata = zipdata.merge(employment_rate_nyc[['zipcode','employment rate']], on='zipcode', how='left')

# --- Current slots per ZIP ---
child_care_regulated_nyc['under5_cap'] = (
    child_care_regulated_nyc['children_capacity']
    + child_care_regulated_nyc['preschool_capacity']
)
# under-5 slots = children_capacity + preschool_capacity
# (infant_capacity and toddler_capacity are 0 in this dataset)

slots_per_zip = child_care_regulated_nyc.groupby('zipcode').agg(
    total_slots    = ('total_capacity', 'sum'),
    under5_slots   = ('under5_cap', 'sum'),
    num_facilities = ('facility_id', 'count')
).reset_index()

zipdata = zipdata.merge(slots_per_zip, on='zipcode', how='left')
zipdata['total_slots']    = zipdata['total_slots'].fillna(0).astype(int)
zipdata['under5_slots']   = zipdata['under5_slots'].fillna(0).astype(int)
zipdata['num_facilities'] = zipdata['num_facilities'].fillna(0).astype(int)

print(f"ZIPs: {len(zipdata)}")
print(f"Total existing slots: {zipdata['total_slots'].sum():,}")
print(f"Total under-5 slots:  {zipdata['under5_slots'].sum():,}")


ZIPs: 180
Total existing slots: 318,149
Total under-5 slots:  69,896


In [ ]:
# --- Classify demand level and identify deserts ---

# High-demand: employment >= 60% OR income <= $60,000
zipdata['high_demand'] = (
    (zipdata['employment rate'] >= 0.60) | (zipdata['average income'] <= 60000)
)
zipdata.loc[
    zipdata['average income'].isna() | zipdata['employment rate'].isna(),
    'high_demand'
] = True
# Missing data -> assume high-demand (conservative).

# Desert threshold
zipdata['desert_threshold'] = np.where(
    zipdata['high_demand'],
    zipdata['pop_0_12'] / 2,     # high-demand: desert if slots <= pop/2
    zipdata['pop_0_12'] / 3      # normal: desert if slots <= pop/3
)

zipdata['is_desert'] = zipdata['total_slots'] <= zipdata['desert_threshold']

# Deficit: how many slots to ADD to escape desert status
# Need total_slots > threshold => target = ceil(threshold) + 1
zipdata['total_deficit'] = np.maximum(
    0, np.ceil(zipdata['desert_threshold']) + 1 - zipdata['total_slots']
).astype(int)

# Under-5 policy: under5_slots >= (2/3) * pop_0_5
zipdata['under5_threshold'] = (2/3) * zipdata['pop_0_5']
zipdata['under5_deficit'] = np.maximum(
    0, np.ceil(zipdata['under5_threshold']) - zipdata['under5_slots']
).astype(int)

print(f"Deserts:            {zipdata['is_desert'].sum()} / {len(zipdata)}")
print(f"High-demand ZIPs:   {zipdata['high_demand'].sum()}")
print(f"Total slot deficit: {zipdata['total_deficit'].sum():,}")
print(f"Under-5 deficit:    {zipdata['under5_deficit'].sum():,}")


Deserts:            162 / 180
High-demand ZIPs:   97
Total slot deficit: 237,370
Under-5 deficit:    276,978


In [ ]:
# --- Build facility-level data (used by both tasks) ---

facility_data = child_care_regulated_nyc[[
    'facility_id','zipcode','total_capacity','under5_cap','latitude','longitude'
]].copy()
facility_data = facility_data.rename(columns={
    'total_capacity': 'nf',     # current slots
    'under5_cap': 'under5_nf'   # current under-5 slots
})

print(f"Facilities: {len(facility_data)}")
print(f"  with nf > 0: {(facility_data['nf'] > 0).sum()}")
print(f"  with nf = 0: {(facility_data['nf'] == 0).sum()}")


Facilities: 7740
  with nf > 0: 7739
  with nf = 0: 1


---
---
# ═══════════════════════════════════════════════════════
# PART A — THE BUDGETING PROBLEM (Task 1: Idealistic Scenario)
# ═══════════════════════════════════════════════════════

**Assumptions:**
1. New facilities can be built **anywhere** — no location restrictions, no distance constraint
2. Expansion limited to **120% of current capacity** OR **500-slot total cap** per facility
3. Expansion cost is **linear**: cost = (20,000/n + 200) × x per slot
4. New under-5 slots cost an extra **$100/slot** for specialized equipment
5. Objective: **minimize total funding** to eliminate all deserts and meet under-5 policy


## A.1 Task 1 Cost Model

The project gives a piecewise cost function:

$$C_{exp}(f) = \begin{cases} c_{slots}(n_f) \cdot x_f & \text{if } x_f < n_f \\\\ (20{,}000 + 200 n_f) \cdot \frac{x_f}{n_f} & \text{if } x_f \geq n_f \end{cases}$$

where $c_{slots}$ is a decreasing function of $n_f$.

**Deriving $c_{slots}$:** At the boundary $x_f = n_f$, both pieces must agree:

$$c_{slots}(n_f) \cdot n_f = (20{,}000 + 200 n_f) \implies c_{slots}(n_f) = \frac{20{,}000}{n_f} + 200$$

This is indeed decreasing in $n_f$ (larger facilities are cheaper to expand per slot).

Both pieces simplify to the same formula, so the cost is simply **linear in $x_f$**:

$$\text{cost} = \left(\frac{20{,}000}{n_f} + 200\right) \cdot x_f$$


In [ ]:
# Task 1 expansion parameters
# Max expansion: min(floor(1.2 * nf), max(500 - nf, 0))
# - Can't add more than 120% of current slots
# - Facility total can't exceed 500 (some already exceed this — they can't expand)
# Cost per slot = 20000/nf + 200

# New facility costs (from Table 1):
NEW_FACILITY = {
    'small':  {'total': 100, 'under5': 50,  'cost': 65000},
    'medium': {'total': 200, 'under5': 100, 'cost': 95000},
    'large':  {'total': 400, 'under5': 200, 'cost': 115000},
}
UNDER5_EQUIP_COST = 100  # $100 per new under-5 slot

# Per-slot cost of each new facility type (including under-5 equipment):
for name, info in NEW_FACILITY.items():
    total_cost = info['cost'] + info['under5'] * UNDER5_EQUIP_COST
    per_slot = total_cost / info['total']
    print(f"  {name:7s}: ${total_cost:,} total = ${per_slot:.2f}/slot")

print()
print("Expansion cost per slot = 20000/nf + 200:")
for nf in [10, 50, 100, 150, 200, 400]:
    c = 20000/nf + 200
    print(f"  nf={nf:>3}: ${c:.2f}/slot")


  small  : $70,000 total = $700.00/slot
  medium : $105,000 total = $525.00/slot
  large  : $135,000 total = $337.50/slot

Expansion cost per slot = 20000/nf + 200:
  nf= 10: $2200.00/slot
  nf= 50: $600.00/slot
  nf=100: $400.00/slot
  nf=150: $333.33/slot
  nf=200: $300.00/slot
  nf=400: $250.00/slot


## A.2 Solve Task 1 — ZIP by ZIP

In [ ]:
# For each ZIP, we find the minimum cost to:
#   1. Eliminate desert status (total slots > threshold)
#   2. Meet under-5 policy (under-5 slots >= 2/3 * pop_0_5)
#
# Strategy (greedy, cheapest-first):
#   - First expand existing facilities (sorted by cost per slot)
#   - Then build new large facilities (cheapest per slot among new builds)
#   - Use smaller facilities only for the tail end

zips_to_solve_t1 = zipdata[
    (zipdata['is_desert']) | (zipdata['under5_deficit'] > 0)
]['zipcode'].values

print(f"ZIPs needing intervention: {len(zips_to_solve_t1)}")

results_t1 = []
total_cost_t1 = 0

for z in sorted(zips_to_solve_t1):
    zrow = zipdata[zipdata['zipcode'] == z].iloc[0]
    total_deficit = int(zrow['total_deficit'])
    u5_deficit    = int(zrow['under5_deficit'])

    facs = facility_data[facility_data['zipcode'] == z].copy().reset_index(drop=True)

    zip_cost = 0
    total_added = 0
    u5_added = 0
    n_expansions = 0
    n_new_builds = 0

    # === STEP 1: Expand existing facilities ===
    # For each facility: max_xf = min(floor(1.2*nf), max(500-nf, 0))
    # Cost per slot = 20000/nf + 200 (plus $100 for under-5 fraction)

    expand_options = []
    for _, fac in facs.iterrows():
        nf = int(fac['nf'])
        if nf == 0:
            continue

        # Expansion cap: can't exceed 120% of current OR 500 total
        max_xf = min(int(np.floor(1.2 * nf)), max(500 - nf, 0))
        if max_xf <= 0:
            continue

        u5_frac = fac['under5_nf'] / nf
        cost_per_slot = (20000 / nf + 200) + UNDER5_EQUIP_COST * u5_frac
        # Base expansion cost + equipment cost for the under-5 portion

        expand_options.append({
            'max_slots': max_xf,
            'cost_per_slot': cost_per_slot,
            'u5_frac': u5_frac
        })

    # Sort cheapest first
    expand_options.sort(key=lambda x: x['cost_per_slot'])

    remaining_total = max(total_deficit, 0)
    remaining_u5 = max(u5_deficit, 0)

    for opt in expand_options:
        if remaining_total <= 0 and remaining_u5 <= 0:
            break

        # How many slots to take from this facility?
        # Take as many as needed (up to max) to reduce deficit
        needed = max(remaining_total, int(np.ceil(remaining_u5 / max(opt['u5_frac'], 0.01))))
        xf = min(opt['max_slots'], needed)
        if xf <= 0:
            continue

        cost = opt['cost_per_slot'] * xf
        u5_gain = int(np.round(xf * opt['u5_frac']))

        zip_cost += cost
        total_added += xf
        u5_added += u5_gain
        remaining_total -= xf
        remaining_u5 -= u5_gain
        n_expansions += 1

    # === STEP 2: Build new facilities ===
    # No location/distance constraints in Task 1!
    # Prefer large (cheapest per slot), then medium, then small.

    while remaining_total > 0 or remaining_u5 > 0:
        need = max(remaining_total, int(remaining_u5 * 2))
        if need >= 200:
            size = 'large'
        elif need >= 100:
            size = 'medium'
        else:
            size = 'small'

        fac_info = NEW_FACILITY[size]
        build_cost = fac_info['cost'] + fac_info['under5'] * UNDER5_EQUIP_COST

        zip_cost += build_cost
        total_added += fac_info['total']
        u5_added += fac_info['under5']
        remaining_total -= fac_info['total']
        remaining_u5 -= fac_info['under5']
        n_new_builds += 1

    total_cost_t1 += zip_cost
    results_t1.append({
        'zipcode': z, 'was_desert': bool(zrow['is_desert']),
        'total_deficit': total_deficit, 'under5_deficit': u5_deficit,
        'total_added': total_added, 'under5_added': u5_added,
        'num_expansions': n_expansions, 'num_new_builds': n_new_builds,
        'zip_cost': zip_cost
    })

print("Task 1 optimization complete.")


ZIPs needing intervention: 180
Task 1 optimization complete.


## A.3 Task 1 Results

In [ ]:
results_t1_df = pd.DataFrame(results_t1)

print("=" * 60)
print("  PART A RESULTS — BUDGETING PROBLEM (Task 1)")
print("=" * 60)
print()
print(f"Total ZIP codes:           {len(zipdata)}")
print(f"ZIPs that were deserts:    {results_t1_df['was_desert'].sum()}")
print(f"ZIPs needing intervention: {len(results_t1_df)}")
print()
print(f"Expansions performed:      {results_t1_df['num_expansions'].sum():,}")
print(f"New facilities built:      {results_t1_df['num_new_builds'].sum():,}")
print(f"Total slots added:         {results_t1_df['total_added'].sum():,}")
print(f"Under-5 slots added:       {results_t1_df['under5_added'].sum():,}")
print()
print("=" * 50)
print(f"  TASK 1 MINIMUM FUNDING: ${total_cost_t1:,.2f}")
print("=" * 50)


  PART A RESULTS — BUDGETING PROBLEM (Task 1)

Total ZIP codes:           180
ZIPs that were deserts:    162
ZIPs needing intervention: 180

Expansions performed:      6,640
New facilities built:      1,128
Total slots added:         740,841
Under-5 slots added:       283,013

  TASK 1 MINIMUM FUNDING: $371,693,426.71


In [ ]:
# Task 1 — Top 10 most expensive ZIPs
print("Task 1 — Top 10 Most Expensive ZIPs:")
print("-" * 70)
for _, row in results_t1_df.nlargest(10, 'zip_cost').iterrows():
    print(f"  ZIP {int(row['zipcode'])}: ${row['zip_cost']:>12,.2f}  |  "
          f"deficit: {int(row['total_deficit']):>5}  |  "
          f"expansions: {int(row['num_expansions']):>3}  |  "
          f"new: {int(row['num_new_builds']):>2}")


Task 1 — Top 10 Most Expensive ZIPs:
----------------------------------------------------------------------
  ZIP 11219: $6,865,535.12  |  deficit: 10780  |  expansions:  91  |  new: 30
  ZIP 11207: $6,143,899.17  |  deficit:  1646  |  expansions: 155  |  new:  7
  ZIP 11208: $6,140,783.72  |  deficit:  4761  |  expansions: 129  |  new: 15
  ZIP 10456: $5,683,256.87  |  deficit:  1215  |  expansions: 177  |  new:  0
  ZIP 11212: $5,383,501.40  |  deficit:  1621  |  expansions: 110  |  new: 11
  ZIP 10457: $5,177,307.46  |  deficit:  1729  |  expansions: 162  |  new:  0
  ZIP 10467: $5,114,141.06  |  deficit:  1980  |  expansions: 160  |  new:  0
  ZIP 11368: $5,008,898.01  |  deficit:  6564  |  expansions:  61  |  new: 21
  ZIP 11236: $4,948,212.64  |  deficit:  2895  |  expansions: 123  |  new:  7
  ZIP 11206: $4,887,495.12  |  deficit:  5385  |  expansions:  57  |  new: 21


In [ ]:
# Task 1 — Summary stats
print("Task 1 Summary:")
print(f"  Avg cost/ZIP:    ${results_t1_df['zip_cost'].mean():,.2f}")
print(f"  Median cost/ZIP: ${results_t1_df['zip_cost'].median():,.2f}")
print(f"  Max cost/ZIP:    ${results_t1_df['zip_cost'].max():,.2f}")
print(f"  Avg slots added: {results_t1_df['total_added'].mean():.0f}")

results_t1_df.to_csv('task1_results_by_zip.csv', index=False)
print("\nSaved: task1_results_by_zip.csv")


Task 1 Summary:
  Avg cost/ZIP:    $2,064,963.48
  Median cost/ZIP: $1,687,443.75
  Max cost/ZIP:    $6,865,535.12
  Avg slots added: 4116

Saved: task1_results_by_zip.csv


---
---
# ═══════════════════════════════════════════════════════
# PART B — REALISTIC CAPACITY EXPANSION & LOCATION (Task 2)
# ═══════════════════════════════════════════════════════

**Changes from Task 1:**
1. Expansion capped at **20%** (not 120%)
2. **3-tier** cost system — higher expansion % = higher rate on ALL added slots
3. New facilities only at **approved potential locations**
4. **0.06-mile minimum distance** between any two facilities within each ZIP

| Expansion Range | Cost Formula |
|---|---|
| 0 < x/n ≤ 10% | (20,000 + 200n) × x/n |
| 10% < x/n ≤ 15% | (20,000 + 400n) × x/n |
| 15% < x/n ≤ 20% | (20,000 + 1,000n) × x/n |

**Key:** The tier rate applies to the **entire** expansion (not just the marginal part).


## B.1 Task 2 Cost Model

In [ ]:
def task2_expansion_cost(nf, xf):
    """
    Task 2 piecewise expansion cost.
    nf = current slots, xf = slots to ADD.
    The tier rate applies to ALL of xf, not just the marginal portion.
    """
    if nf == 0 or xf <= 0:
        return 0
    ratio = xf / nf
    if ratio <= 0.10:
        return (20000 + 200 * nf) * ratio   # Tier 1: cheapest
    elif ratio <= 0.15:
        return (20000 + 400 * nf) * ratio   # Tier 2: medium
    elif ratio <= 0.20:
        return (20000 + 1000 * nf) * ratio  # Tier 3: expensive
    else:
        return float('inf')                  # >20% not allowed

# Verify with project example: nf=100, xf=12 -> ratio=0.12 -> Tier 2
ex = task2_expansion_cost(100, 12)
print(f"Example: nf=100, xf=12 -> ${ex:,.0f}")
print(f"  Check: (20000 + 400*100) * 12/100 = {(20000+40000)*0.12:,.0f}  ✓")


Example: nf=100, xf=12 -> $7,200
  Check: (20000 + 400*100) * 12/100 = 7,200  ✓


## B.2 Distance Constraint Preprocessing

In [ ]:
# Filter out potential locations too close to existing facilities.
# Then find conflict pairs among remaining locations.

MIN_DIST = 0.06  # miles (~317 feet)
print(f"Min distance: {MIN_DIST} miles\n")

# --- Step 1: Remove locations too close to existing facilities ---
feasible_locations = {}  # zip -> list of (lat, lon, idx)

for z in sorted(pop_zips):
    existing = facility_data[facility_data['zipcode'] == z]
    ex_coords = existing[['latitude','longitude']].values

    pot_locs = potential_locations_nyc[potential_locations_nyc['zipcode'] == z]
    if len(pot_locs) == 0:
        feasible_locations[z] = []
        continue

    pot_coords = pot_locs[['latitude','longitude']].values
    feasible = []
    for i, (plat, plon) in enumerate(pot_coords):
        if len(ex_coords) > 0:
            dists = haversine(plat, plon, ex_coords[:,0], ex_coords[:,1])
            if np.any(dists < MIN_DIST):
                continue  # too close to an existing facility
        feasible.append((plat, plon, pot_locs.index[i]))
    feasible_locations[z] = feasible

total_feasible = sum(len(v) for v in feasible_locations.values())
print(f"Potential locations: {len(potential_locations_nyc):,}")
print(f"Feasible (pass distance check): {total_feasible:,}")
print(f"Removed: {len(potential_locations_nyc) - total_feasible:,}")


Min distance: 0.06 miles

Potential locations: 31,100
Feasible (pass distance check): 27,972
Removed: 3,128


In [ ]:
# --- Step 2: Find conflict pairs among feasible locations ---
# Two feasible locations in the same ZIP that are < 0.06 mi apart
# can't both be selected.

conflict_pairs = {}
for z in sorted(pop_zips):
    locs = feasible_locations[z]
    conflicts = []
    for i in range(len(locs)):
        for j in range(i+1, len(locs)):
            d = haversine(locs[i][0], locs[i][1], locs[j][0], locs[j][1])
            if d < MIN_DIST:
                conflicts.append((i, j))
    conflict_pairs[z] = conflicts

print(f"Conflict pairs: {sum(len(v) for v in conflict_pairs.values()):,}")


Conflict pairs: 72,467


## B.3 Solve Task 2 — ZIP by ZIP

In [ ]:
# For each ZIP:
#   1. Expand existing facilities (3-tier costs, max 20%, cheapest first)
#   2. Build new facilities at feasible locations (respecting distance constraint)

zips_to_solve_t2 = zipdata[
    (zipdata['is_desert']) | (zipdata['under5_deficit'] > 0)
]['zipcode'].values

print(f"ZIPs needing intervention: {len(zips_to_solve_t2)}")

results_t2 = []
total_cost_t2 = 0

for z_idx, z in enumerate(sorted(zips_to_solve_t2)):
    zrow = zipdata[zipdata['zipcode'] == z].iloc[0]
    total_deficit = int(zrow['total_deficit'])
    u5_deficit    = int(zrow['under5_deficit'])

    facs = facility_data[facility_data['zipcode'] == z].copy().reset_index(drop=True)
    feas_locs = feasible_locations.get(z, [])
    n_locs = len(feas_locs)

    zip_cost = 0
    total_added = 0
    u5_added = 0
    n_expansions = 0
    n_new_builds = 0

    # === STEP 1: Expand existing (3-tier, max 20%) ===
    # Each facility has up to 3 MUTUALLY EXCLUSIVE options (Tier 1/2/3).
    # Tier rate applies to the ENTIRE expansion, so Tier 2 costs more per
    # slot than Tier 1 (even for the first 10%).
    # We pick the cheapest options across all facilities.

    expand_options = []
    for _, fac in facs.iterrows():
        nf = int(fac['nf'])
        if nf == 0:
            continue
        u5_frac = fac['under5_nf'] / nf

        # Tier 1: up to 10%
        t1 = int(np.floor(nf * 0.10))
        if t1 > 0:
            cost = (20000 + 200*nf) * (t1/nf)
            u5g = int(np.round(t1 * u5_frac))
            tc = cost + u5g * UNDER5_EQUIP_COST
            expand_options.append({'fac': fac.name, 'slots': t1, 'tier': 1,
                'cost': tc, 'cps': tc/t1, 'u5g': u5g})

        # Tier 2: up to 15%
        t2 = int(np.floor(nf * 0.15))
        if t2 > t1:
            cost = (20000 + 400*nf) * (t2/nf)
            u5g = int(np.round(t2 * u5_frac))
            tc = cost + u5g * UNDER5_EQUIP_COST
            expand_options.append({'fac': fac.name, 'slots': t2, 'tier': 2,
                'cost': tc, 'cps': tc/t2, 'u5g': u5g})

        # Tier 3: up to 20%
        t3 = int(np.floor(nf * 0.20))
        if t3 > t2:
            cost = (20000 + 1000*nf) * (t3/nf)
            u5g = int(np.round(t3 * u5_frac))
            tc = cost + u5g * UNDER5_EQUIP_COST
            expand_options.append({'fac': fac.name, 'slots': t3, 'tier': 3,
                'cost': tc, 'cps': tc/t3, 'u5g': u5g})

    expand_options.sort(key=lambda x: x['cps'])  # cheapest per slot first

    used_facs = set()
    remaining_total = max(total_deficit, 0)
    remaining_u5 = max(u5_deficit, 0)

    for opt in expand_options:
        if remaining_total <= 0 and remaining_u5 <= 0:
            break
        if opt['fac'] in used_facs:
            continue
        used_facs.add(opt['fac'])
        zip_cost += opt['cost']
        total_added += opt['slots']
        u5_added += opt['u5g']
        remaining_total -= opt['slots']
        remaining_u5 -= opt['u5g']
        n_expansions += 1

    # === STEP 2: Build new facilities (with distance constraints) ===
    if remaining_total > 0 or remaining_u5 > 0:
        adj = {i: set() for i in range(n_locs)}
        for (i, j) in conflict_pairs.get(z, []):
            adj[i].add(j)
            adj[j].add(i)

        blocked = set()
        while remaining_total > 0 or remaining_u5 > 0:
            need = max(remaining_total, int(remaining_u5 * 2))
            if need >= 200:
                size = 'large'
            elif need >= 100:
                size = 'medium'
            else:
                size = 'small'

            fi = NEW_FACILITY[size]
            bc = fi['cost'] + fi['under5'] * UNDER5_EQUIP_COST

            placed = False
            for loc_i in range(n_locs):
                if loc_i in blocked:
                    continue
                zip_cost += bc
                total_added += fi['total']
                u5_added += fi['under5']
                remaining_total -= fi['total']
                remaining_u5 -= fi['under5']
                n_new_builds += 1
                blocked.add(loc_i)
                for cj in adj.get(loc_i, set()):
                    blocked.add(cj)
                placed = True
                break
            if not placed:
                print(f"  ⚠ ZIP {z}: no more feasible locations!")
                break

    total_cost_t2 += zip_cost
    results_t2.append({
        'zipcode': z, 'was_desert': bool(zrow['is_desert']),
        'total_deficit': total_deficit, 'under5_deficit': u5_deficit,
        'total_added': total_added, 'under5_added': u5_added,
        'num_expansions': n_expansions, 'num_new_builds': n_new_builds,
        'zip_cost': zip_cost
    })
    if (z_idx+1) % 30 == 0:
        print(f"  ... {z_idx+1}/{len(zips_to_solve_t2)} ZIPs ...")

print(f"\nAll {len(zips_to_solve_t2)} ZIPs processed.")


ZIPs needing intervention: 180
  ... 30/180 ZIPs ...
  ... 60/180 ZIPs ...
  ... 90/180 ZIPs ...
  ... 120/180 ZIPs ...
  ... 150/180 ZIPs ...
  ... 180/180 ZIPs ...

All 180 ZIPs processed.


## B.4 Task 2 Results

In [ ]:
results_t2_df = pd.DataFrame(results_t2)

print("=" * 60)
print("  PART B RESULTS — REALISTIC EXPANSION & LOCATION (Task 2)")
print("=" * 60)
print()
print(f"Total ZIP codes:           {len(zipdata)}")
print(f"ZIPs that were deserts:    {results_t2_df['was_desert'].sum()}")
print(f"ZIPs needing intervention: {len(results_t2_df)}")
print()
print(f"Expansions performed:      {results_t2_df['num_expansions'].sum():,}")
print(f"New facilities built:      {results_t2_df['num_new_builds'].sum():,}")
print(f"Total slots added:         {results_t2_df['total_added'].sum():,}")
print(f"Under-5 slots added:       {results_t2_df['under5_added'].sum():,}")
print()
print("=" * 50)
print(f"  TASK 2 MINIMUM FUNDING: ${total_cost_t2:,.2f}")
print("=" * 50)


  PART B RESULTS — REALISTIC EXPANSION & LOCATION (Task 2)

Total ZIP codes:           180
ZIPs that were deserts:    162
ZIPs needing intervention: 180

Expansions performed:      7,731
New facilities built:      1,447
Total slots added:         584,913
Under-5 slots added:       284,400

  TASK 2 MINIMUM FUNDING: $209,749,583.36


In [ ]:
# Task 2 — Top 10 most expensive ZIPs
print("Task 2 — Top 10 Most Expensive ZIPs:")
print("-" * 70)
for _, row in results_t2_df.nlargest(10, 'zip_cost').iterrows():
    print(f"  ZIP {int(row['zipcode'])}: ${row['zip_cost']:>12,.2f}  |  "
          f"deficit: {int(row['total_deficit']):>5}  |  "
          f"expansions: {int(row['num_expansions']):>3}  |  "
          f"new: {int(row['num_new_builds']):>2}")


Task 2 — Top 10 Most Expensive ZIPs:
----------------------------------------------------------------------
  ZIP 11219: $4,898,211.43  |  deficit: 10780  |  expansions:  91  |  new: 35
  ZIP 11230: $3,427,821.43  |  deficit:  4980  |  expansions:  28  |  new: 25
  ZIP 11368: $3,402,116.73  |  deficit:  6564  |  expansions:  61  |  new: 24
  ZIP 11206: $3,211,946.22  |  deficit:  5385  |  expansions:  57  |  new: 23
  ZIP 11208: $3,140,232.40  |  deficit:  4761  |  expansions: 130  |  new: 21
  ZIP 11204: $3,104,986.66  |  deficit:  5492  |  expansions:  50  |  new: 22
  ZIP 11223: $2,998,027.66  |  deficit:  6293  |  expansions:  19  |  new: 22
  ZIP 11220: $2,478,508.48  |  deficit:  6094  |  expansions:  44  |  new: 18
  ZIP 11233: $2,459,835.33  |  deficit:  2991  |  expansions:  62  |  new: 17
  ZIP 11385: $2,445,275.96  |  deficit:  4300  |  expansions:  66  |  new: 17


In [ ]:
# Task 2 — Summary stats
print("Task 2 Summary:")
print(f"  Avg cost/ZIP:    ${results_t2_df['zip_cost'].mean():,.2f}")
print(f"  Median cost/ZIP: ${results_t2_df['zip_cost'].median():,.2f}")
print(f"  Max cost/ZIP:    ${results_t2_df['zip_cost'].max():,.2f}")
print(f"  Avg slots added: {results_t2_df['total_added'].mean():.0f}")

results_t2_df.to_csv('task2_results_by_zip.csv', index=False)
print("\nSaved: task2_results_by_zip.csv")


Task 2 Summary:
  Avg cost/ZIP:    $1,165,275.46
  Median cost/ZIP: $977,578.36
  Max cost/ZIP:    $4,898,211.43
  Avg slots added: 3250

Saved: task2_results_by_zip.csv


---
---
# ═══════════════════════════════════════════════════════
# COMPARISON — Task 1 vs Task 2
# ═══════════════════════════════════════════════════════


In [ ]:
print("=" * 60)
print("         SIDE-BY-SIDE COMPARISON")
print("=" * 60)
print()
print(f"{'Metric':<30s} {'Task 1':>15s} {'Task 2':>15s}")
print("-" * 60)
print(f"{'Minimum Funding':<30s} {'${:,.0f}'.format(total_cost_t1):>15s} {'${:,.0f}'.format(total_cost_t2):>15s}")
print(f"{'Expansions':<30s} {results_t1_df['num_expansions'].sum():>15,} {results_t2_df['num_expansions'].sum():>15,}")
print(f"{'New facilities':<30s} {results_t1_df['num_new_builds'].sum():>15,} {results_t2_df['num_new_builds'].sum():>15,}")
print(f"{'Total slots added':<30s} {results_t1_df['total_added'].sum():>15,} {results_t2_df['total_added'].sum():>15,}")
print(f"{'Under-5 slots added':<30s} {results_t1_df['under5_added'].sum():>15,} {results_t2_df['under5_added'].sum():>15,}")
print()
pct = (total_cost_t2 - total_cost_t1) / total_cost_t1 * 100
print(f"Task 2 costs {pct:+.1f}% {'more' if pct > 0 else 'less'} than Task 1.")
print("This reflects the realistic constraints: limited expansion,")
print("higher tiered costs, and location/distance restrictions.")


         SIDE-BY-SIDE COMPARISON

Metric                                  Task 1          Task 2
------------------------------------------------------------
Minimum Funding                   $371,693,427    $209,749,583
Expansions                               6,640           7,731
New facilities                           1,128           1,447
Total slots added                      740,841         584,913
Under-5 slots added                    283,013         284,400

Task 2 costs -43.6% less than Task 1.
This reflects the realistic constraints: limited expansion,
higher tiered costs, and location/distance restrictions.
